# Prepare df_ssp for plots and stats

In [ ]:
# import all packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

### Import output data for all SSPs

In [ ]:
from scripts.plots_stats.get_df_ssp import get_df_ssp
df_ssp = get_df_ssp()

In [ ]:
df_ssp[['GEOID', 'NAMELSAD', 'city type', 'REGION', 'CensusPop_20','length_m_perCap_2020_mean']].sort_values('length_m_perCap_2020_mean').head(10)

### Percent cities with increasing burden from 2020 to 2100

In [ ]:
print('Cities with an increasing burden at each scenario ====')
print("For buildings=== mean, min , max ")
print(df_ssp[df_ssp['volume_m3_perCap_2020_mean'] < df_ssp['volume_m3_perCap_2100_mean']].shape[0] / df_ssp['volume_m3_perCap_2100_mean'].shape[0],\
     df_ssp[df_ssp['volume_m3_perCap_2020_min'] < df_ssp['volume_m3_perCap_2100_min']].shape[0] / df_ssp['volume_m3_perCap_2100_min'].shape[0],  \
        df_ssp[df_ssp['volume_m3_perCap_2020_max'] < df_ssp['volume_m3_perCap_2100_max']].shape[0] / df_ssp['volume_m3_perCap_2100_max'].shape[0])
print("For roadways=== mean, min , max ")
print(df_ssp[df_ssp['length_m_perCap_2020_mean'] < df_ssp['length_m_perCap_2100_mean']].shape[0] / df_ssp['volume_m3_perCap_2100_mean'].shape[0], \
df_ssp[df_ssp['length_m_perCap_2020_min'] < df_ssp['length_m_perCap_2100_min']].shape[0] / df_ssp['volume_m3_perCap_2100_min'].shape[0], \
df_ssp[df_ssp['length_m_perCap_2020_max'] < df_ssp['length_m_perCap_2100_max']].shape[0] / df_ssp['volume_m3_perCap_2100_max'].shape[0] )

### Regional decriptive stats by city type

In [ ]:
df_ssp.groupby(['city type','REGION'])[['volume_m3_perCap_2020_mean','length_m_perCap_2020_mean']].median().sort_values(['city type','volume_m3_perCap_2020_mean']).round(2)

### Cities with less than 1000 m/person roadway

In [ ]:
df_ssp[df_ssp['length_m_perCap_2020_mean']<1000].sort_values('length_m_perCap_2020_mean').tail(100)['length_m_perCap_2100_mean'].describe().round()

In [ ]:
df_ssp['length_m_perCap_2020_mean'].quantile([.5,.95]), df_ssp[df_ssp['length_m_perCap_2020_mean']<1000]['length_m_perCap_2020_mean'].quantile([.5,.95])

### Cities with over 1000 m/person roadway

In [ ]:
df_ssp[df_ssp['length_m_perCap_2020_mean']>1000].sort_values('length_m_perCap_2020_mean')[['GEOID','NAMELSAD', 'citytype_at_2030', 'median_income']].head(50)

### Cities with less than 100 m3/person built volume

In [ ]:
df_ssp[df_ssp['volume_m3_perCap_2020_mean']<100].sort_values('length_m_perCap_2020_mean')[['GEOID','NAMELSAD', 'citytype_at_2030', 'median_income']].head(50)

### Cities with over 2000 m3/person built volume and their landuse

In [ ]:
df_ssp[df_ssp['volume_m3_perCap_2020_mean']>2000][['NAMELSAD', 'REGION','city type', 'median_income']] 

### Variation with region and city type

In [ ]:
hue = 'city type'
hu_order = ['urban', 'suburban', 'periurban', 'rural']
citytypes = ['Northeast', 'Midwest', 'West', 'South']
fig, axs = plt.subplots(nrows = 2, ncols =2, figsize = (12,8))
ax = axs.flatten()
for i, city in enumerate(citytypes):
    avocado = df_ssp[df_ssp['REGION'] == city]
    sns.histplot(avocado, x = 'length_m_perCap_2020_mean', log_scale=True, bins = 45,linewidth=0.01, alpha = .40, hue =hue, hue_order = hu_order, kde=True, kde_kws = {'cut': 0}, ax=ax[i],
                 palette = ['red', 'lightseagreen', 'dimgrey', 'darkkhaki'], ) # ['red', 'orange', 'green', 'tan'],)
    ax[i].set_ylabel("No of cities", fontsize=12)
    ax[i].set_title(city)
    if i >1:
        ax[i].set_xlabel("Per capita roadway length (m)", fontsize=12)
    else:
        ax[i].set_xlabel("", fontsize=10)

In [ ]:
df_ssp[['volume_m3_perCap_2020_mean', 'length_m_perCap_2020_mean']].describe()

In [ ]:
print(f"Cities with less than 100 cubic meter/person RBUV:==  {df_ssp[df_ssp['volume_m3_perCap_2020_mean'] < 100].shape[0]}")
print(f"Cities with less than 10 m/person roadway:==          {df_ssp[df_ssp['length_m_perCap_2020_mean'] < 10].shape[0]}")
print(f"Cities with over 1000 m/person roadway:==             {df_ssp[df_ssp['length_m_perCap_2020_mean'] > 1000].shape[0]}")

In [ ]:
df_ssp['RBUV_Burden_2050'].value_counts(), df_ssp['RL_Burden_2050'].value_counts(), df_ssp['RBUV_Burden_2100'].value_counts(), df_ssp['RL_Burden_2100'].value_counts()

In [ ]:
df_ssp.groupby('STATEFP')[[ 'volume_m3_perCap_2020_mean','volume_m3_perCap_2050_mean','volume_m3_perCap_2100_mean','length_m_perCap_2020_mean','length_m_perCap_2050_mean', 
                       'length_m_perCap_2100_mean']].median().round(1).to_csv(r'outputfiles\csvs\state_level_median.csv')

In [ ]:
df_ssp[(df_ssp['CensusPop_20']<=10000)].groupby(['STATEFP'])[['volume_m3_perCap_2020_mean','volume_m3_perCap_2050_mean','volume_m3_perCap_2100_ssp2','length_m_perCap_2020_ssp2',
                                                              'length_m_perCap_2050_ssp2', 'length_m_perCap_2100_ssp2', 'added_RBUV_2020_2050', 'added_RL_2020_2050',
                         'added_RBUV_2050_2100', 'added_RL_2050_2100']].median().round(2).to_csv(r'outputfiles\csvs\state_level_median_below_10000.csv')

### Constantly increasing burden

In [ ]:
condition = (df_ssp['volume_m3_perCap_2020_ssp2'] < df_ssp['volume_m3_perCap_2030_ssp2']) & (df_ssp['volume_m3_perCap_2030_ssp2'] < df_ssp['volume_m3_perCap_2040_ssp2']) & \
(df_ssp['volume_m3_perCap_2040_ssp2'] < df_ssp['volume_m3_perCap_2050_ssp2']) & (df_ssp['volume_m3_perCap_2050_ssp2'] < df_ssp['volume_m3_perCap_2060_ssp2']) & \
(df_ssp['volume_m3_perCap_2060_ssp2'] < df_ssp['volume_m3_perCap_2070_ssp2']) & (df_ssp['volume_m3_perCap_2070_ssp2'] < df_ssp['volume_m3_perCap_2080_ssp2']) & \
(df_ssp['volume_m3_perCap_2080_ssp2'] < df_ssp['volume_m3_perCap_2090_ssp2']) & (df_ssp['volume_m3_perCap_2090_ssp2'] < df_ssp['volume_m3_perCap_2100_ssp2'])
print(f"cities that face constantly increasing burden for RBUV {np.round(df_ssp[condition].shape[0]*100/df_ssp.shape[0],2)}")

condition = (df_ssp['length_m_perCap_2020_ssp2'] < df_ssp['length_m_perCap_2030_ssp2']) & (df_ssp['length_m_perCap_2030_ssp2'] < df_ssp['length_m_perCap_2040_ssp2']) & \
(df_ssp['length_m_perCap_2040_ssp2'] < df_ssp['length_m_perCap_2050_ssp2']) & (df_ssp['length_m_perCap_2050_ssp2'] < df_ssp['length_m_perCap_2060_ssp2']) & \
(df_ssp['length_m_perCap_2060_ssp2'] < df_ssp['length_m_perCap_2070_ssp2']) & (df_ssp['length_m_perCap_2070_ssp2'] < df_ssp['length_m_perCap_2080_ssp2']) & \
(df_ssp['length_m_perCap_2080_ssp2'] < df_ssp['length_m_perCap_2090_ssp2']) & (df_ssp['length_m_perCap_2090_ssp2'] < df_ssp['length_m_perCap_2100_ssp2'])
print(f"cities that face constantly increasing burden for RL {np.round(df_ssp[condition].shape[0]*100/df_ssp.shape[0],2)}")

In [ ]:
df_ssp.filter(regex='volume').columns
print("Percent cities with increasing built volumes thereby increasing burden")

column_nmaes1 = ['volume_m3_perCap_2020_ssp2','volume_m3_perCap_2030_ssp2', 'volume_m3_perCap_2040_ssp2','volume_m3_perCap_2050_ssp2', 'volume_m3_perCap_2060_ssp2',
                'volume_m3_perCap_2070_ssp2', 'volume_m3_perCap_2080_ssp2', 'volume_m3_perCap_2090_ssp2']
column_nmaes2 = ['volume_m3_perCap_2030_ssp2', 'volume_m3_perCap_2040_ssp2','volume_m3_perCap_2050_ssp2', 'volume_m3_perCap_2060_ssp2',
                'volume_m3_perCap_2070_ssp2', 'volume_m3_perCap_2080_ssp2', 'volume_m3_perCap_2090_ssp2', 'volume_m3_perCap_2100_ssp2']
for col1, col2 in zip(column_nmaes1, column_nmaes2):
    print(np.round(df_ssp[df_ssp[col2] > df_ssp[col1]].shape[0]*100/df_ssp.shape[0],2))

print("Percent cities with increasing roadways thereby increasing burden")

column_nmaes1 = ['length_m_perCap_2020_ssp2','length_m_perCap_2030_ssp2', 'length_m_perCap_2040_ssp2','length_m_perCap_2050_ssp2', 
                 'length_m_perCap_2060_ssp2','length_m_perCap_2070_ssp2', 'length_m_perCap_2080_ssp2', 'length_m_perCap_2090_ssp2']
column_nmaes2 = ['length_m_perCap_2030_ssp2', 'length_m_perCap_2040_ssp2','length_m_perCap_2050_ssp2', 'length_m_perCap_2060_ssp2',
                 'length_m_perCap_2070_ssp2', 'length_m_perCap_2080_ssp2', 'length_m_perCap_2090_ssp2', 'length_m_perCap_2100_ssp2']
for col1, col2 in zip(column_nmaes1, column_nmaes2):
    print(np.round(df_ssp[df_ssp[col2] > df_ssp[col1]].shape[0]*100/df_ssp.shape[0],2))

### Constandtly decreasing burden

In [ ]:
condition = (df_ssp['volume_m3_perCap_2020_ssp2'] > df_ssp['volume_m3_perCap_2030_ssp2']) & (df_ssp['volume_m3_perCap_2030_ssp2'] > df_ssp['volume_m3_perCap_2040_ssp2']) & \
            (df_ssp['volume_m3_perCap_2040_ssp2'] > df_ssp['volume_m3_perCap_2050_ssp2']) & (df_ssp['volume_m3_perCap_2050_ssp2'] > df_ssp['volume_m3_perCap_2060_ssp2']) & \
            (df_ssp['volume_m3_perCap_2060_ssp2'] > df_ssp['volume_m3_perCap_2070_ssp2']) & (df_ssp['volume_m3_perCap_2070_ssp2'] > df_ssp['volume_m3_perCap_2080_ssp2']) & \
            (df_ssp['volume_m3_perCap_2080_ssp2'] > df_ssp['volume_m3_perCap_2090_ssp2']) & (df_ssp['volume_m3_perCap_2090_ssp2'] > df_ssp['volume_m3_perCap_2100_ssp2'])
print(f"cities that face constantly decreasing burden for RBUV {np.round(df_ssp[condition].shape[0]*100/df_ssp.shape[0],2)}")

condition = (df_ssp['length_m_perCap_2020_ssp2'] > df_ssp['length_m_perCap_2030_ssp2']) & (df_ssp['length_m_perCap_2030_ssp2'] > df_ssp['length_m_perCap_2040_ssp2']) & \
            (df_ssp['length_m_perCap_2040_ssp2'] > df_ssp['length_m_perCap_2050_ssp2']) & (df_ssp['length_m_perCap_2050_ssp2'] > df_ssp['length_m_perCap_2060_ssp2']) & \
            (df_ssp['length_m_perCap_2060_ssp2'] > df_ssp['length_m_perCap_2070_ssp2']) & (df_ssp['length_m_perCap_2070_ssp2'] > df_ssp['length_m_perCap_2080_ssp2']) & \
            (df_ssp['length_m_perCap_2080_ssp2'] > df_ssp['length_m_perCap_2090_ssp2']) & (df_ssp['length_m_perCap_2090_ssp2'] > df_ssp['length_m_perCap_2100_ssp2'])
print(f"cities that face constantly decreasing burden for RL {np.round(df_ssp[condition].shape[0]*100/df_ssp.shape[0],2)}")

In [ ]:
df_ssp.filter(regex='volume').columns
print("Percent cities with decreasing built volumes")

column_nmaes1 = ['volume_m3_perCap_2020_ssp2','volume_m3_perCap_2030_ssp2', 'volume_m3_perCap_2040_ssp2','volume_m3_perCap_2050_ssp2', 'volume_m3_perCap_2060_ssp2',
                'volume_m3_perCap_2070_ssp2', 'volume_m3_perCap_2080_ssp2', 'volume_m3_perCap_2090_ssp2']
column_nmaes2 = ['volume_m3_perCap_2030_ssp2', 'volume_m3_perCap_2040_ssp2','volume_m3_perCap_2050_ssp2', 'volume_m3_perCap_2060_ssp2',
                'volume_m3_perCap_2070_ssp2', 'volume_m3_perCap_2080_ssp2', 'volume_m3_perCap_2090_ssp2', 'volume_m3_perCap_2100_ssp2']
for col1, col2 in zip(column_nmaes1, column_nmaes2):
    print(np.round(df_ssp[df_ssp[col2] < df_ssp[col1]].shape[0]*100/df_ssp.shape[0],2))

print("Percent cities with decreasing roadways")

column_nmaes1 = ['length_m_perCap_2020_ssp2','length_m_perCap_2030_ssp2', 'length_m_perCap_2040_ssp2','length_m_perCap_2050_ssp2', 
                 'length_m_perCap_2060_ssp2','length_m_perCap_2070_ssp2', 'length_m_perCap_2080_ssp2', 'length_m_perCap_2090_ssp2']
column_nmaes2 = ['length_m_perCap_2030_ssp2', 'length_m_perCap_2040_ssp2','length_m_perCap_2050_ssp2', 'length_m_perCap_2060_ssp2',
                 'length_m_perCap_2070_ssp2', 'length_m_perCap_2080_ssp2', 'length_m_perCap_2090_ssp2', 'length_m_perCap_2100_ssp2']
for col1, col2 in zip(column_nmaes1, column_nmaes2):
    print(np.round(df_ssp[df_ssp[col2] < df_ssp[col1]].shape[0]*100/df_ssp.shape[0],2))

In [ ]:
df_ssp[['GEOID', 'NAMELSAD', 'city type', 'REGION', 'ALAND','label', 'future trend from SSP 2',
    'CensusPop_20','ssp22030', 'ssp22040', 'ssp22050','ssp22060',
       'ssp22070', 'ssp22080', 'ssp22090', 'ssp22100' ,'volume_m3_perCap_2020_ssp2', 'volume_m3_perCap_2030_ssp2',
       'volume_m3_perCap_2040_ssp2', 'volume_m3_perCap_2050_ssp2',
       'volume_m3_perCap_2060_ssp2', 'volume_m3_perCap_2070_ssp2',
       'volume_m3_perCap_2080_ssp2', 'volume_m3_perCap_2090_ssp2',
       'volume_m3_perCap_2100_ssp2','length_m_perCap_2020_ssp2',
       'length_m_perCap_2030_ssp2', 'length_m_perCap_2040_ssp2',
       'length_m_perCap_2050_ssp2', 'length_m_perCap_2060_ssp2',
       'length_m_perCap_2070_ssp2', 'length_m_perCap_2080_ssp2',
       'length_m_perCap_2090_ssp2', 'length_m_perCap_2100_ssp2',
       'weighted_HU_density_sqmi', 'citytype_at_2030', 'citytype_at_2040', 
       'citytype_at_2050', 'citytype_at_2060', 'citytype_at_2070', 
       'citytype_at_2080', 'citytype_at_2090', 'citytype_at_2100', 
       'RBUV_Burden_2050', 'RBUV_Burden_2100','RL_Burden_2050', 'RL_Burden_2100']].to_csv(r'outputfiles\csvs\df_ssp2_clean.csv')